In [1]:
import os
import pandas as pd
import numpy as np 

In [2]:
RAW_FOLDER = r"C:\Users\HP\ISRO_Project\Rainfall\Raw_Data"
print(os.listdir(RAW_FOLDER)[:10]) 

['Rainfall_ind2010_rfp25.grd', 'Rainfall_ind2011_rfp25.grd', 'Rainfall_ind2012_rfp25.grd', 'Rainfall_ind2013_rfp25.grd', 'Rainfall_ind2014_rfp25.grd', 'Rainfall_ind2015_rfp25.grd', 'Rainfall_ind2016_rfp25.grd', 'Rainfall_ind2017_rfp25.grd', 'Rainfall_ind2018_rfp25.grd', 'Rainfall_ind2019_rfp25.grd']


In [3]:
RAW_FOLDER = r"C:\Users\HP\ISRO_Project\Rainfall\Raw_Data"
for year in range(2010, 2026):
    file_path = os.path.join(
        RAW_FOLDER,
        f"Rainfall_ind{year}_rfp25.grd"
    )

    if os.path.exists(file_path):
        rain_data = np.fromfile(file_path, dtype=np.float32)

        days = rain_data.size // (129 * 135)

        print(
            f"Year: {year} | "
            f"Shape: {rain_data.shape} | "
            f"Days: {days}"
        )
    else:
        print(f"Missing file: {year}")

Year: 2010 | Shape: (6356475,) | Days: 365
Year: 2011 | Shape: (6356475,) | Days: 365
Year: 2012 | Shape: (6373890,) | Days: 366
Year: 2013 | Shape: (6356475,) | Days: 365
Year: 2014 | Shape: (6356475,) | Days: 365
Year: 2015 | Shape: (6356475,) | Days: 365
Year: 2016 | Shape: (6373890,) | Days: 366
Year: 2017 | Shape: (6356475,) | Days: 365
Year: 2018 | Shape: (6356475,) | Days: 365
Year: 2019 | Shape: (6356475,) | Days: 365
Year: 2020 | Shape: (6373890,) | Days: 366
Year: 2021 | Shape: (6356475,) | Days: 365
Year: 2022 | Shape: (6356475,) | Days: 365
Year: 2023 | Shape: (6356475,) | Days: 365
Year: 2024 | Shape: (6373890,) | Days: 366
Year: 2025 | Shape: (6356475,) | Days: 365


In [4]:
rain_data = np.fromfile(
    "Raw_Data/Rainfall_ind2010_rfp25.grd",
    dtype=np.float32
)
print(rain_data.shape)

(6356475,)


In [5]:
rain_data = np.fromfile(
    "Raw_Data/Rainfall_ind2010_rfp25.grd",
    dtype=np.float32
)
days = rain_data.size // (129 * 135)
rainfall = rain_data.reshape(days, 129, 135)
print(rainfall.shape)

(365, 129, 135)


In [6]:
rainfall[rainfall == -999] = np.nan
print(np.nanmin(rainfall))
print(np.nanmax(rainfall))

0.0
494.71695


In [7]:
latitudes = np.arange(6.5, 6.5 + 129 * 0.25, 0.25)
longitudes = np.arange(66.5, 66.5 + 135 * 0.25, 0.25)
print(latitudes[:5])
print(latitudes[-5:])
print(longitudes[:5])
print(longitudes[-5:])

[6.5  6.75 7.   7.25 7.5 ]
[37.5  37.75 38.   38.25 38.5 ]
[66.5  66.75 67.   67.25 67.5 ]
[ 99.    99.25  99.5   99.75 100.  ]


In [8]:
print("Latitude:", latitudes[50])
print("Longitude:", longitudes[50])
print("Rainfall:", rainfall[0, 50, 50])

Latitude: 19.0
Longitude: 79.0
Rainfall: 0.0


In [9]:
dates = pd.date_range(
    start="2010-01-01",
    periods=days,
    freq="D"
)

records = []

for day in range(days):
    for i in range(0, len(latitudes), 4):  
        for j in range(0, len(longitudes), 4):

            # 4x4 block ke liye 
            block = rainfall[day, i:i+4, j:j+4]
            if np.isnan(block).all():
                value = np.nan
            else:
                value = np.nanmean(block)

            # Record add karne ke liye 
            records.append([
                dates[day],
                latitudes[i] + 1.0,      # 6.5 -> 7.5
                longitudes[j] + 1.0,     # 66.5 -> 67.5
                value
            ])

# DataFrame bnaya hu 
df_rain = pd.DataFrame(
    records,
    columns=[
        "Date",
        "Latitude",
        "Longitude",
        "Rainfall"
    ]
)

In [10]:
print(df_rain.head())
print(df_rain.shape)

        Date  Latitude  Longitude  Rainfall
0 2010-01-01       7.5       67.5       NaN
1 2010-01-01       7.5       68.5       NaN
2 2010-01-01       7.5       69.5       NaN
3 2010-01-01       7.5       70.5       NaN
4 2010-01-01       7.5       71.5       NaN
(409530, 4)


In [11]:
print(df_rain["Latitude"].min(), df_rain["Latitude"].max())
print(df_rain["Longitude"].min(), df_rain["Longitude"].max())
print(df_rain["Latitude"].nunique())
print(df_rain["Longitude"].nunique())

7.5 39.5
67.5 100.5
33
34


In [12]:
dates = pd.date_range(
    start="2010-01-01",
    periods=days,
    freq="D"
)

records = []

# 31*31 grid banya hu 
for day in range(days):
    for i in range(4, 128, 4):      
        for j in range(4, 128, 4): 

            block = rainfall[day, i-4:i, j-4:j]

            if np.isnan(block).all():
                value = np.nan
            else:
                value = np.nanmean(block)

            records.append([
                dates[day],
                latitudes[i],
                longitudes[j],
                value
            ])

df_rain = pd.DataFrame(
    records,
    columns=[
        "Date",
        "Latitude",
        "Longitude",
        "Rainfall"
    ]
)

In [13]:
print(df_rain.head())
print(df_rain.shape)
print(df_rain["Latitude"].min(), df_rain["Latitude"].max())
print(df_rain["Longitude"].min(), df_rain["Longitude"].max())
print(df_rain["Latitude"].nunique())
print(df_rain["Longitude"].nunique())

        Date  Latitude  Longitude  Rainfall
0 2010-01-01       7.5       67.5       NaN
1 2010-01-01       7.5       68.5       NaN
2 2010-01-01       7.5       69.5       NaN
3 2010-01-01       7.5       70.5       NaN
4 2010-01-01       7.5       71.5       NaN
(350765, 4)
7.5 37.5
67.5 97.5
31
31


In [14]:
df_rain.to_csv(
    r"C:\Users\HP\ISRO_Project\Rainfall\CSV_Data\df_rain_2010.csv",
    index=False
)
print("df_rain_2010.csv saved successfully!")

df_rain_2010.csv saved successfully!


In [15]:
print(df_rain.dropna().head())
print(df_rain.describe())
print(df_rain.isna().sum())

         Date  Latitude  Longitude  Rainfall
41 2010-01-01       8.5       77.5       0.0
42 2010-01-01       8.5       78.5       0.0
71 2010-01-01       9.5       76.5       0.0
72 2010-01-01       9.5       77.5       0.0
73 2010-01-01       9.5       78.5       0.0
                      Date       Latitude      Longitude       Rainfall
count               350765  350765.000000  350765.000000  134685.000000
mean   2010-07-02 00:00:00      22.500000      82.500000       3.734577
min    2010-01-01 00:00:00       7.500000      67.500000       0.000000
25%    2010-04-02 00:00:00      14.500000      74.500000       0.000000
50%    2010-07-02 00:00:00      22.500000      82.500000       0.000000
75%    2010-10-01 00:00:00      30.500000      90.500000       2.249833
max    2010-12-31 00:00:00      37.500000      97.500000     381.978973
std                    NaN       8.944285       8.944285      10.707592
Date              0
Latitude          0
Longitude         0
Rainfall     216080
dt

In [16]:
print(" File validation completed")
print("All files detected successfully")
print("Ready for batch processing")

 File validation completed
All files detected successfully
Ready for batch processing


In [17]:
RAW_FOLDER = r"C:\Users\HP\ISRO_Project\Rainfall\Raw_Data"
OUTPUT_FOLDER = r"C:\Users\HP\ISRO_Project\Rainfall\CSV_Data"

In [18]:
latitudes = np.arange(6.5, 6.5 + 129 * 0.25, 0.25)
longitudes = np.arange(66.5, 66.5 + 135 * 0.25, 0.25)

In [19]:
for year in range(2010, 2026):
    print(f"\n===== Processing {year} =====")
    file_path = os.path.join(
        RAW_FOLDER,
        f"Rainfall_ind{year}_rfp25.grd"
    )

    rain_data = np.fromfile(file_path, dtype=np.float32)

    # Automatic days calclualte karne ke liye 
    days = rain_data.size // (129 * 135)

    # Reshape
    rainfall = rain_data.reshape(days, 129, 135)

    # Missing value ke liye 
    rainfall[rainfall == -999] = np.nan
    #date ke liye 
    dates = pd.date_range(
        start=f"{year}-01-01",
        periods=days,
        freq="D"
    )

    records = []

    # grid (31 × 31) ke liye 
    for day in range(days):
        for i in range(4, 128, 4):
            for j in range(4, 128, 4):

                block = rainfall[day, i-4:i, j-4:j]

                if np.isnan(block).all():
                    value = np.nan
                else:
                    value = np.nanmean(block)

                records.append([
                    dates[day],
                    latitudes[i],
                    longitudes[j],
                    value
                ])

    # DataFrame ke liye
    df_rain = pd.DataFrame(
        records,
        columns=[
            "Date",
            "Latitude",
            "Longitude",
            "Rainfall"
        ]
    )

    # Save
    save_path = os.path.join(
        OUTPUT_FOLDER,
        f"df_rain_{year}.csv"
    )

    df_rain.to_csv(save_path, index=False)

    print(f"Saved: df_rain_{year}.csv")
    print(f"Shape: {df_rain.shape}")

print("\nAll Rainfall files processed successfully!")


===== Processing 2010 =====
Saved: df_rain_2010.csv
Shape: (350765, 4)

===== Processing 2011 =====
Saved: df_rain_2011.csv
Shape: (350765, 4)

===== Processing 2012 =====
Saved: df_rain_2012.csv
Shape: (351726, 4)

===== Processing 2013 =====
Saved: df_rain_2013.csv
Shape: (350765, 4)

===== Processing 2014 =====
Saved: df_rain_2014.csv
Shape: (350765, 4)

===== Processing 2015 =====
Saved: df_rain_2015.csv
Shape: (350765, 4)

===== Processing 2016 =====
Saved: df_rain_2016.csv
Shape: (351726, 4)

===== Processing 2017 =====
Saved: df_rain_2017.csv
Shape: (350765, 4)

===== Processing 2018 =====
Saved: df_rain_2018.csv
Shape: (350765, 4)

===== Processing 2019 =====
Saved: df_rain_2019.csv
Shape: (350765, 4)

===== Processing 2020 =====
Saved: df_rain_2020.csv
Shape: (351726, 4)

===== Processing 2021 =====
Saved: df_rain_2021.csv
Shape: (350765, 4)

===== Processing 2022 =====
Saved: df_rain_2022.csv
Shape: (350765, 4)

===== Processing 2023 =====
Saved: df_rain_2023.csv
Shape: (350

In [22]:
df_check = pd.read_csv(
    r"C:\Users\HP\ISRO_Project\Rainfall\CSV_Data\df_rain_2025.csv")

In [23]:
print(df_check.head())
print(df_check.shape)

         Date  Latitude  Longitude  Rainfall
0  2025-01-01       7.5       67.5       NaN
1  2025-01-01       7.5       68.5       NaN
2  2025-01-01       7.5       69.5       NaN
3  2025-01-01       7.5       70.5       NaN
4  2025-01-01       7.5       71.5       NaN
(350765, 4)


In [24]:
print(df_check["Latitude"].min(), df_check["Latitude"].max())
print(df_check["Longitude"].min(), df_check["Longitude"].max())

7.5 37.5
67.5 97.5


In [25]:
print(df_check.describe())
print(df_check.isna().sum())

            Latitude      Longitude       Rainfall
count  350765.000000  350765.000000  134685.000000
mean       22.500000      82.500000       3.672949
std         8.944285       8.944285      10.257903
min         7.500000      67.500000       0.000000
25%        14.500000      74.500000       0.000000
50%        22.500000      82.500000       0.000000
75%        30.500000      90.500000       2.221981
max        37.500000      97.500000     310.014587
Date              0
Latitude          0
Longitude         0
Rainfall     216080
dtype: int64
